# 04 — Final results and interpretation

**Purpose.** Reconstruct the seven selected specifications, fit them on the same internal model-training subset, evaluate the final test set, and create interpretation tables.

**Inputs.** Training/test data, selected model settings, and the feature dictionary.

**Outputs.** Final-test metrics, final predictions, classification reports, logistic coefficients, and tree feature importances.

**What you will learn.** How final-test evaluation is kept separate from model selection and how the fitted models are interpreted.

**Run first.** Run Notebooks 01-03 first.

## Imports and paths

In [1]:
from __future__ import annotations

import ast
import json
import math
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, PercentFormatter
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

current_path = Path.cwd().resolve()
if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

assert project_root.name == "ml-finance-bankruptcy-analysis", (
    f"Unexpected project root: {project_root}"
)

DATA_DIR = project_root / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = project_root / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"
REPORT_DIR = OUTPUTS_DIR / "report"
PAPER_TABLES_DIR = OUTPUTS_DIR / "paper" / "tables"
PAPER_FIGURES_DIR = OUTPUTS_DIR / "paper" / "figures"

for path in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    REPORT_DIR,
    PAPER_TABLES_DIR,
    PAPER_FIGURES_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)


## Project constants

In [2]:
RAW_DATA_PATH = RAW_DATA_DIR / "american_bankruptcy.csv"
MODEL_DATASET_PATH = PROCESSED_DATA_DIR / "model_dataset.csv"
TRAIN_DATA_PATH = PROCESSED_DATA_DIR / "train.csv"
TEST_DATA_PATH = PROCESSED_DATA_DIR / "test.csv"

COMPANY_COLUMN = "company_name"
RAW_TARGET_COLUMN = "status_label"
TARGET_COLUMN = "failed"
YEAR_COLUMN = "year"

ALIVE_LABEL = "alive"
FAILED_LABEL = "failed"

RANDOM_STATE = 42
OUTER_TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20
LOGISTIC_C_GRID = [0.01, 0.1, 1.0, 10.0]
PCA_COMPONENT_GRID = [2, 3, 5, 8, 10, 12, 15, 18]

EXPECTED_MODEL_ORDER = [
    "Majority-class baseline",
    "Logistic Regression",
    "L1 Regularized Logistic Regression",
    "L2 Regularized Logistic Regression",
    "Decision Tree",
    "Random Forest",
    "Gradient Boosting",
]

PREDICTION_TABLE_COLUMNS = [
    "model",
    COMPANY_COLUMN,
    YEAR_COLUMN,
    "actual_failed",
    "predicted_failed",
    "probability_failed",
]
PREDICTION_PROBABILITY_FLOAT_FORMAT = "%.11f"


## Feature names and descriptions

In [3]:
FEATURE_NAME_MAP = {
    "X1": "Current assets",
    "X2": "Cost of goods sold",
    "X3": "Depreciation and amortization",
    "X4": "EBITDA",
    "X5": "Inventory",
    "X6": "Net income",
    "X7": "Total receivables",
    "X8": "Market value",
    "X9": "Net sales",
    "X10": "Total assets",
    "X11": "Total long-term debt",
    "X12": "EBIT",
    "X13": "Gross profit",
    "X14": "Total current liabilities",
    "X15": "Retained earnings",
    "X16": "Total revenue",
    "X17": "Total liabilities",
    "X18": "Total operating expenses",
}

FEATURE_DESCRIPTION_MAP = {
    "X1": "Assets expected to be sold, converted into cash, or used within one year.",
    "X2": "Direct costs related to producing or selling the firm's goods and services.",
    "X3": "Depreciation of tangible assets and amortization of intangible assets.",
    "X4": "Earnings before interest, taxes, depreciation, and amortization.",
    "X5": "Goods and raw materials held by the firm for production or sale.",
    "X6": "Profit after expenses and costs have been deducted from revenue.",
    "X7": "Money owed to the firm by customers for delivered goods or services.",
    "X8": "Market capitalization or market value of the publicly traded company.",
    "X9": "Gross sales minus returns, allowances, and discounts.",
    "X10": "Total assets owned or controlled by the company.",
    "X11": "Debt obligations due after more than one year.",
    "X12": "Earnings before interest and taxes.",
    "X13": "Profit after subtracting costs related to manufacturing and selling.",
    "X14": "Short-term obligations due within one year.",
    "X15": "Accumulated profits retained in the business after dividends and losses.",
    "X16": "Total income from sales before subtracting expenses.",
    "X17": "Total debts and obligations owed to outside parties.",
    "X18": "Expenses incurred through normal business operations.",
}

KEY_FEATURES_FOR_EDA = ["X8", "X6", "X11", "X1", "X17", "X15"]
KEY_MODELS_FOR_THRESHOLD_ANALYSIS = [
    "Logistic Regression",
    "Random Forest",
    "Gradient Boosting",
]


## Figure style helpers

In [4]:
MODEL_COLORS = {
    "Majority-class baseline": "#8c8c8c",
    "Logistic Regression": "#4c78a8",
    "Random Forest": "#59a14f",
    "Gradient Boosting": "#f28e2b",
}
OUTCOME_COLORS = {"detected": "#59a14f", "missed": "#e15759", "false_alarm": "#f28e2b"}
DIRECTION_COLORS = {"positive": "#c44e52", "negative": "#4c78a8"}
METRIC_COLORS = {
    "Precision": "#4c78a8",
    "Recall": "#e15759",
    "F1": "#59a14f",
    "ROC-AUC": "#4c78a8",
    "PR-AUC": "#f28e2b",
    "Failed F1": "#59a14f",
}
BASELINE_COLOR = "#8c8c8c"


def apply_project_style() -> None:
    """Apply the common Matplotlib style used across project figures."""
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "axes.titlesize": 12,
            "axes.labelsize": 10,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 8.5,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "grid.color": "#d9d9d9",
            "grid.linewidth": 0.8,
            "lines.linewidth": 2.0,
            "lines.markersize": 5.5,
        }
    )


def style_axis(ax, *, grid_axis: str = "y", percent_y: bool = False, integer_x: bool = False) -> None:
    """Add consistent grid and tick formatting to one axis."""
    ax.grid(axis=grid_axis, linestyle="--", alpha=0.25)
    if percent_y:
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    if integer_x:
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))


def save_figure(fig, output_path: Path) -> None:
    """Save a Matplotlib figure with the project's standard export settings."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


## Data validation and feature helpers

In [5]:
def validate_required_columns(data: pd.DataFrame) -> None:
    """Confirm that the raw dataset has the identifier, year, and target columns."""
    missing = {COMPANY_COLUMN, RAW_TARGET_COLUMN, YEAR_COLUMN}.difference(data.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")


def validate_target_labels(data: pd.DataFrame) -> None:
    """Confirm that the raw target contains only the expected alive/failed labels."""
    observed = set(data[RAW_TARGET_COLUMN].dropna().unique())
    unexpected = observed.difference({ALIVE_LABEL, FAILED_LABEL})
    if unexpected:
        raise ValueError(f"Unexpected target labels found: {sorted(unexpected)}")


def identify_feature_columns(data: pd.DataFrame) -> list[str]:
    """Return numeric financial variables from the raw dataset, excluding identifiers."""
    excluded = {COMPANY_COLUMN, RAW_TARGET_COLUMN, YEAR_COLUMN}
    return [
        column
        for column in data.columns
        if column not in excluded and pd.api.types.is_numeric_dtype(data[column])
    ]


def get_feature_columns(data: pd.DataFrame, include_year: bool = False) -> list[str]:
    """Return modelling predictors while keeping identifiers and target out of X."""
    excluded = {COMPANY_COLUMN, TARGET_COLUMN}
    if not include_year:
        excluded.add(YEAR_COLUMN)
    return [column for column in data.columns if column not in excluded]


def split_features_target(data: pd.DataFrame, include_year: bool = False) -> tuple[pd.DataFrame, pd.Series]:
    """Split a model-ready table into predictor matrix and binary target."""
    feature_columns = get_feature_columns(data, include_year=include_year)
    return data[feature_columns].copy(), data[TARGET_COLUMN].copy()


## Company split helpers

In [6]:
def create_company_level_split(
    data: pd.DataFrame,
    test_size: float = OUTER_TEST_SIZE,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split rows by company so a firm cannot appear in both samples."""
    missing = {COMPANY_COLUMN, TARGET_COLUMN}.difference(data.columns)
    if missing:
        raise ValueError(f"Missing required split columns: {sorted(missing)}")

    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(
        splitter.split(data, y=data[TARGET_COLUMN], groups=data[COMPANY_COLUMN])
    )
    return data.iloc[train_idx].copy(), data.iloc[test_idx].copy()


def check_no_company_overlap(left: pd.DataFrame, right: pd.DataFrame) -> bool:
    """Return True when two samples have disjoint company identifiers."""
    left_companies = set(left[COMPANY_COLUMN].unique())
    right_companies = set(right[COMPANY_COLUMN].unique())
    return left_companies.isdisjoint(right_companies)


def create_split_summary(
    full_data: pd.DataFrame,
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
) -> pd.DataFrame:
    """Summarize row counts, company counts, and failure rates for a split."""
    def summarize(name: str, data: pd.DataFrame) -> dict[str, float | int | str]:
        return {
            "split": name,
            "n_rows": int(len(data)),
            "n_companies": int(data[COMPANY_COLUMN].nunique()),
            "n_failed_rows": int(data[TARGET_COLUMN].sum()),
            "n_alive_rows": int(len(data) - data[TARGET_COLUMN].sum()),
            "failure_rate": float(data[TARGET_COLUMN].mean()),
        }

    summary = pd.DataFrame(
        [summarize("full", full_data), summarize("train", train_data), summarize("test", test_data)]
    )
    summary["company_share"] = summary["n_companies"] / int(full_data[COMPANY_COLUMN].nunique())
    summary["row_share"] = summary["n_rows"] / int(len(full_data))
    return summary


## Evaluation helpers

In [7]:
def get_probability_failed(model, x: pd.DataFrame) -> np.ndarray:
    """Return predicted probabilities for the failed class."""
    return model.predict_proba(x)[:, 1]


def evaluate_binary_classifier(
    model_name: str,
    y_true: pd.Series,
    y_pred: np.ndarray,
    probability_failed: np.ndarray,
) -> dict[str, float | int | str]:
    """Calculate the common classification metrics for one fitted model."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, probability_failed),
        "pr_auc": average_precision_score(y_true, probability_failed),
        "precision_failed": precision_score(y_true, y_pred, zero_division=0),
        "recall_failed": recall_score(y_true, y_pred, zero_division=0),
        "f1_failed": f1_score(y_true, y_pred, zero_division=0),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }


def create_classification_report_table(
    model_name: str,
    y_true: pd.Series,
    y_pred: np.ndarray,
) -> pd.DataFrame:
    """Convert scikit-learn's classification report into a flat table."""
    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["alive", "failed"],
        output_dict=True,
        zero_division=0,
    )
    rows = []
    for label, metrics in report.items():
        if isinstance(metrics, dict):
            rows.append({"model": model_name, "class_or_average": label, **metrics})
    return pd.DataFrame(rows)


def create_prediction_table(
    data: pd.DataFrame,
    model_name: str,
    y_pred: np.ndarray,
    probability_failed: np.ndarray,
) -> pd.DataFrame:
    """Create an identifier-preserving prediction table for one model."""
    table = pd.DataFrame(
        {
            "model": model_name,
            COMPANY_COLUMN: data[COMPANY_COLUMN].to_numpy(),
            YEAR_COLUMN: data[YEAR_COLUMN].to_numpy(),
            "actual_failed": data[TARGET_COLUMN].to_numpy(),
            "predicted_failed": y_pred,
            "probability_failed": probability_failed,
        }
    )
    return table[PREDICTION_TABLE_COLUMNS]


def save_prediction_table(predictions: pd.DataFrame, output_path: Path) -> None:
    """Write prediction probabilities with deterministic decimal formatting."""
    predictions[PREDICTION_TABLE_COLUMNS].to_csv(
        output_path,
        index=False,
        float_format=PREDICTION_PROBABILITY_FLOAT_FORMAT,
    )


## Model builders and selection helpers

In [8]:
def build_majority_class_baseline() -> DummyClassifier:
    """Build the benchmark model that always predicts the most frequent class."""
    return DummyClassifier(strategy="most_frequent")


def build_logistic_pipeline(
    penalty: str = "l2",
    c_value: float = 1.0,
    class_weight: str | dict | None = "balanced",
) -> Pipeline:
    """Build a scaled Logistic Regression pipeline with the original settings."""
    if penalty not in {"l1", "l2"}:
        raise ValueError("Only 'l1' and 'l2' penalties are supported in this project.")

    l1_ratio = 1.0 if penalty == "l1" else 0.0
    model = LogisticRegression(
        C=c_value,
        l1_ratio=l1_ratio,
        solver="saga",
        class_weight=class_weight,
        max_iter=5_000,
        random_state=RANDOM_STATE,
    )
    return Pipeline(steps=[("scaler", StandardScaler()), ("model", model)])


def build_interpretable_logit() -> Pipeline:
    """Build the fixed Logistic Regression benchmark used for interpretation."""
    return build_logistic_pipeline(penalty="l2", c_value=1.0)


def select_regularized_logit(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
    penalty: str,
    c_grid: list[float],
) -> tuple[Pipeline, float, float]:
    """Select L1 or L2 Logistic Regression by validation PR-AUC."""
    best_model, best_c, best_score = None, None, -1.0
    for c_value in c_grid:
        candidate = build_logistic_pipeline(penalty=penalty, c_value=c_value)
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model, best_c, best_score = candidate, c_value, score
    if best_model is None or best_c is None:
        raise RuntimeError("No Logistic Regression model was selected.")
    return best_model, best_c, best_score


def build_decision_tree(max_depth: int | None = 3, min_samples_leaf: int = 100) -> DecisionTreeClassifier:
    """Build a class-weighted Decision Tree with the source project settings."""
    return DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )


def build_random_forest(
    n_estimators: int = 300,
    max_depth: int | None = 5,
    min_samples_leaf: int = 50,
    max_features: str | None = "sqrt",
) -> RandomForestClassifier:
    """Build a Random Forest using the original class weights and seed."""
    return RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def build_gradient_boosting(
    n_estimators: int = 150,
    learning_rate: float = 0.05,
    max_depth: int = 2,
    min_samples_leaf: int = 100,
) -> GradientBoostingClassifier:
    """Build a Gradient Boosting classifier with the source project settings."""
    return GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=RANDOM_STATE,
    )


def select_decision_tree(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
) -> tuple[DecisionTreeClassifier, dict[str, int], float]:
    """Search the original Decision Tree grid and keep the best PR-AUC model."""
    grid = {"max_depth": [2, 3, 4, 5], "min_samples_leaf": [50, 100, 200]}
    best_model, best_params, best_score = None, None, -1.0
    for max_depth, min_samples_leaf in product(grid["max_depth"], grid["min_samples_leaf"]):
        candidate = build_decision_tree(max_depth=max_depth, min_samples_leaf=min_samples_leaf)
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {"max_depth": max_depth, "min_samples_leaf": min_samples_leaf}
            best_score = score
    if best_model is None or best_params is None:
        raise RuntimeError("No Decision Tree model was selected.")
    return best_model, best_params, best_score


def select_random_forest(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
) -> tuple[RandomForestClassifier, dict[str, object], float]:
    """Search the original Random Forest grid and keep the best PR-AUC model."""
    grid = {
        "n_estimators": [300],
        "max_depth": [4, 6, None],
        "min_samples_leaf": [50, 100],
        "max_features": ["sqrt"],
    }
    best_model, best_params, best_score = None, None, -1.0
    for n_estimators, max_depth, min_samples_leaf, max_features in product(
        grid["n_estimators"], grid["max_depth"], grid["min_samples_leaf"], grid["max_features"]
    ):
        candidate = build_random_forest(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
        )
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
                "max_features": max_features,
            }
            best_score = score
    if best_model is None or best_params is None:
        raise RuntimeError("No Random Forest model was selected.")
    return best_model, best_params, best_score


def select_gradient_boosting(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_valid: pd.DataFrame,
    y_valid: pd.Series,
) -> tuple[GradientBoostingClassifier, dict[str, object], float]:
    """Search the original Gradient Boosting grid using balanced sample weights."""
    grid = {
        "n_estimators": [100, 150],
        "learning_rate": [0.03, 0.05],
        "max_depth": [2, 3],
        "min_samples_leaf": [100],
    }
    sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
    best_model, best_params, best_score = None, None, -1.0
    for n_estimators, learning_rate, max_depth, min_samples_leaf in product(
        grid["n_estimators"], grid["learning_rate"], grid["max_depth"], grid["min_samples_leaf"]
    ):
        candidate = build_gradient_boosting(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
        )
        candidate.fit(x_train, y_train, sample_weight=sample_weight)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {
                "n_estimators": n_estimators,
                "learning_rate": learning_rate,
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
            }
            best_score = score
    if best_model is None or best_params is None:
        raise RuntimeError("No Gradient Boosting model was selected.")
    return best_model, best_params, best_score


## Model reconstruction helpers

In [9]:
def parse_selected_parameters(value: object) -> dict[str, object]:
    """Parse the saved parameter string from `model_specification.csv`."""
    if pd.isna(value) or value == "":
        return {}

    text = str(value)
    if text.startswith("{"):
        return ast.literal_eval(text)

    params: dict[str, object] = {}
    for piece in text.split(","):
        if "=" in piece:
            key, val = piece.strip().split("=", 1)
            params[key] = float(val) if key == "C" else val
    return params


def build_model_from_spec(model_name: str, selected_parameters: object):
    """Recreate one model from its saved specification without using model binaries."""
    params = parse_selected_parameters(selected_parameters)
    if model_name == "Majority-class baseline":
        return build_majority_class_baseline()
    if model_name == "Logistic Regression":
        return build_interpretable_logit()
    if model_name == "L1 Regularized Logistic Regression":
        return build_logistic_pipeline(penalty="l1", c_value=params["C"])
    if model_name == "L2 Regularized Logistic Regression":
        return build_logistic_pipeline(penalty="l2", c_value=params["C"])
    if model_name == "Decision Tree":
        return build_decision_tree(**params)
    if model_name == "Random Forest":
        return build_random_forest(**params)
    if model_name == "Gradient Boosting":
        return build_gradient_boosting(**params)
    raise ValueError(f"Unknown model: {model_name}")


## Load final-evaluation inputs

In [10]:
train_full = pd.read_csv(TRAIN_DATA_PATH)
test_data = pd.read_csv(TEST_DATA_PATH)
model_specification = pd.read_csv(TABLES_DIR / "model_specification.csv").fillna("")
feature_dictionary = pd.read_csv(TABLES_DIR / "feature_dictionary.csv")

display(model_specification)
display(pd.DataFrame({"sample": ["outer_train", "outer_test"], "rows": [len(train_full), len(test_data)], "companies": [train_full[COMPANY_COLUMN].nunique(), test_data[COMPANY_COLUMN].nunique()]}))

,model,model_type,model_family,selected_parameters,selection_metric,validation_pr_auc_during_selection
0,Majority-class baseline,DummyClassifier,baseline,,,
1,Logistic Regression,LogisticRegression,logistic_regression,"penalty=l2, C=1.0",fixed benchmark,
2,L1 Regularized Logistic Regression,LogisticRegression,logistic_regression,"penalty=l1, C=0.1",validation PR-AUC,0.148295
3,L2 Regularized Logistic Regression,LogisticRegression,logistic_regression,"penalty=l2, C=0.1",validation PR-AUC,0.14873
4,Decision Tree,DecisionTreeClassifier,tree_based,"{'max_depth': 5, 'min_samples_leaf': 50}",validation PR-AUC,0.146024
5,Random Forest,RandomForestClassifier,tree_based,"{'n_estimators': 300, 'max_depth': None, 'min_...",validation PR-AUC,0.154216
6,Gradient Boosting,GradientBoostingClassifier,tree_based,"{'n_estimators': 150, 'learning_rate': 0.05, '...",validation PR-AUC,0.174674


,sample,rows,companies
0,outer_train,62988,7176
1,outer_test,15694,1795


## Recreate the internal fitting subset

The validation rows are not added back before final-test evaluation. This preserves the original research design.

In [11]:
model_train, validation = create_company_level_split(
    train_full,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
)
assert check_no_company_overlap(model_train, validation), "Internal reconstruction split contains overlap."
assert set(model_train[COMPANY_COLUMN]).isdisjoint(set(test_data[COMPANY_COLUMN])), "Outer test companies must not enter fitting data."

x_train, y_train = split_features_target(model_train)
x_test, y_test = split_features_target(test_data)

display(pd.DataFrame({"sample": ["model_train", "held_out_validation", "outer_test"], "rows": [len(model_train), len(validation), len(test_data)], "companies": [model_train[COMPANY_COLUMN].nunique(), validation[COMPANY_COLUMN].nunique(), test_data[COMPANY_COLUMN].nunique()]}))

,sample,rows,companies
0,model_train,50323,5740
1,held_out_validation,12665,1436
2,outer_test,15694,1795


## Rebuild and fit the seven selected models

In [12]:
models = {}
fit_log = []
for _, spec in model_specification.iterrows():
    model = build_model_from_spec(spec["model"], spec["selected_parameters"])
    if spec["model"] == "Gradient Boosting":
        weights = compute_sample_weight(class_weight="balanced", y=y_train)
        model.fit(x_train, y_train, sample_weight=weights)
    else:
        model.fit(x_train, y_train)
    models[spec["model"]] = model
    fit_log.append({"model": spec["model"], "selected_parameters": spec["selected_parameters"]})

assert list(models.keys()) == EXPECTED_MODEL_ORDER, "Final model order changed."
display(pd.DataFrame(fit_log))

,model,selected_parameters
0,Majority-class baseline,
1,Logistic Regression,"penalty=l2, C=1.0"
2,L1 Regularized Logistic Regression,"penalty=l1, C=0.1"
3,L2 Regularized Logistic Regression,"penalty=l2, C=0.1"
4,Decision Tree,"{'max_depth': 5, 'min_samples_leaf': 50}"
5,Random Forest,"{'n_estimators': 300, 'max_depth': None, 'min_..."
6,Gradient Boosting,"{'n_estimators': 150, 'learning_rate': 0.05, '..."


## Evaluate the untouched final test set

In [13]:
metric_rows, report_tables, prediction_tables = [], [], []
for model_name, model in models.items():
    y_pred = model.predict(x_test)
    probability_failed = get_probability_failed(model, x_test)
    assert len(probability_failed) == len(test_data), f"{model_name} probability length mismatch."
    assert pd.Series(probability_failed).between(0, 1).all(), f"{model_name} probabilities outside [0, 1]."
    metric_rows.append(evaluate_binary_classifier(model_name, y_test, y_pred, probability_failed))
    report_tables.append(create_classification_report_table(model_name, y_test, y_pred))
    prediction_tables.append(create_prediction_table(test_data, model_name, y_pred, probability_failed))

final_test_metrics = pd.DataFrame(metric_rows)
final_reports = pd.concat(report_tables, ignore_index=True)
final_predictions = pd.concat(prediction_tables, ignore_index=True)
for _, row in final_test_metrics.iterrows():
    assert row[["true_negative", "false_positive", "false_negative", "true_positive"]].sum() == len(test_data), f"Confusion counts inconsistent for {row.model}."

display(final_test_metrics)

,model,accuracy,balanced_accuracy,roc_auc,pr_auc,precision_failed,recall_failed,f1_failed,true_negative,false_positive,false_negative,true_positive
0,Majority-class baseline,0.925003,0.500000,0.500000,0.074997,0.000000,0.000000,0.000000,14517,0,1177,0
1,Logistic Regression,0.360138,0.601040,0.690046,0.152647,0.095095,0.884452,0.171726,4611,9906,136,1041
2,L1 Regularized Logistic Regression,0.357971,0.601821,0.691740,0.153603,0.095169,0.888700,0.171926,4572,9945,131,1046
3,L2 Regularized Logistic Regression,0.358481,0.601706,0.691603,0.152995,0.095164,0.887850,0.171903,4581,9936,132,1045
4,Decision Tree,0.599847,0.580711,0.633065,0.122088,0.102384,0.558199,0.173031,8757,5760,520,657
5,Random Forest,0.813623,0.612337,0.698898,0.159223,0.167933,0.375531,0.232082,12327,2190,735,442
6,Gradient Boosting,0.678540,0.638082,0.687586,0.158443,0.132180,0.590484,0.216006,9954,4563,482,695


## Save final-test outputs

In [14]:
final_test_metrics.to_csv(TABLES_DIR / "final_test_metrics.csv", index=False)
final_reports.to_csv(TABLES_DIR / "final_test_classification_reports.csv", index=False)
save_prediction_table(final_predictions, TABLES_DIR / "final_test_predictions.csv")

display(pd.DataFrame({"output": ["final_test_metrics.csv", "final_test_classification_reports.csv", "final_test_predictions.csv"], "exists": [(TABLES_DIR / "final_test_metrics.csv").exists(), (TABLES_DIR / "final_test_classification_reports.csv").exists(), (TABLES_DIR / "final_test_predictions.csv").exists()]}))

,output,exists
0,final_test_metrics.csv,True
1,final_test_classification_reports.csv,True
2,final_test_predictions.csv,True


## Logistic Regression coefficients

In [15]:
feature_columns = get_feature_columns(model_train)
logit_model = models["Logistic Regression"]
coefficients = logit_model.named_steps["model"].coef_[0]
logistic_coefficients = pd.DataFrame({"feature": feature_columns, "coefficient": coefficients})
logistic_coefficients["absolute_coefficient"] = logistic_coefficients["coefficient"].abs()
logistic_coefficients["direction"] = logistic_coefficients["coefficient"].apply(lambda value: "higher predicted failure risk" if value > 0 else "lower predicted failure risk")
logistic_coefficients = logistic_coefficients.merge(feature_dictionary[["feature", "readable_name", "description"]], on="feature", how="left").sort_values("absolute_coefficient", ascending=False)
logistic_coefficients.to_csv(TABLES_DIR / "logistic_coefficients.csv", index=False)
display(logistic_coefficients.head(12))

,feature,coefficient,absolute_coefficient,direction,readable_name,description
0,X1,-1.911823,1.911823,lower predicted failure risk,Current assets,"Assets expected to be sold, converted into cas..."
7,X8,-1.812354,1.812354,lower predicted failure risk,Market value,Market capitalization or market value of the p...
13,X14,0.928426,0.928426,higher predicted failure risk,Total current liabilities,Short-term obligations due within one year.
11,X12,-0.723310,0.723310,lower predicted failure risk,EBIT,Earnings before interest and taxes.
2,X3,0.459962,0.459962,higher predicted failure risk,Depreciation and amortization,Depreciation of tangible assets and amortizati...
12,X13,0.442180,0.442180,higher predicted failure risk,Gross profit,Profit after subtracting costs related to manu...
14,X15,-0.405712,0.405712,lower predicted failure risk,Retained earnings,Accumulated profits retained in the business a...
3,X4,-0.345581,0.345581,lower predicted failure risk,EBITDA,"Earnings before interest, taxes, depreciation,..."
4,X5,0.338916,0.338916,higher predicted failure risk,Inventory,Goods and raw materials held by the firm for p...
9,X10,0.335959,0.335959,higher predicted failure risk,Total assets,Total assets owned or controlled by the company.


## Tree-based feature importances

In [16]:
rows = []
for model_name in ["Decision Tree", "Random Forest", "Gradient Boosting"]:
    model = models[model_name]
    assert hasattr(model, "feature_importances_"), f"{model_name} must expose feature_importances_."
    for feature, importance in zip(feature_columns, model.feature_importances_, strict=False):
        rows.append({"model": model_name, "feature": feature, "importance": importance})

tree_feature_importance = pd.DataFrame(rows).merge(feature_dictionary[["feature", "readable_name", "description"]], on="feature", how="left").sort_values(["model", "importance"], ascending=[True, False])
tree_feature_importance.to_csv(TABLES_DIR / "tree_feature_importance.csv", index=False)
display(tree_feature_importance.groupby("model").head(8))

,model,feature,importance,readable_name,description
5,Decision Tree,X6,0.282137,Net income,Profit after expenses and costs have been dedu...
7,Decision Tree,X8,0.212353,Market value,Market capitalization or market value of the p...
10,Decision Tree,X11,0.152623,Total long-term debt,Debt obligations due after more than one year.
4,Decision Tree,X5,0.081512,Inventory,Goods and raw materials held by the firm for p...
6,Decision Tree,X7,0.079111,Total receivables,Money owed to the firm by customers for delive...
1,Decision Tree,X2,0.051094,Cost of goods sold,Direct costs related to producing or selling t...
2,Decision Tree,X3,0.046446,Depreciation and amortization,Depreciation of tangible assets and amortizati...
13,Decision Tree,X14,0.030489,Total current liabilities,Short-term obligations due within one year.
43,Gradient Boosting,X8,0.179253,Market value,Market capitalization or market value of the p...
41,Gradient Boosting,X6,0.167773,Net income,Profit after expenses and costs have been dedu...
